In [1]:
def metagenePfofileToST(inputFile,outputFile,cdslength=400,expression=50):
    #########################################################
    # prepare gene expression and length
    with open(inputFile,'r') as input:
        gene_infoDict = {}
        for line in input:
            fileds = line.split()
            gene_name = fileds[0].split('|')[0]
            pos = int(fileds[1])
            start = int(fileds[0].split('|')[2])
            end = int(fileds[0].split('|')[3])
            cdsLength = end - start + 1
            # geneLength = int(fileds[0].split('|')[4])
            density = float(fileds[2])
            # filter CDS > 400 nt gene
            if cdsLength > cdslength:
                # key
                key = ':'.join([gene_name,str(cdsLength)])
                gene_infoDict.setdefault(key,0)
                if start <= pos <= end:
                    gene_infoDict[key] += density
                else:
                    pass
            else:
                pass

    # filter CDS expression > 50
    filtedGeneDict = {}
    for key,val in gene_infoDict.items():
        if val > expression:
            meanNorm = val/int(key.split(':')[1])
            filtedGeneDict[key] = [val,meanNorm]
        else:
            pass
    #########################################################
    # Meta-gene analysis from start codon
    mylist = range(-50, 1501)
    rangeDict = dict([i, 0] for i in mylist)
    countDict = dict([i, 0] for i in mylist)

    # open file
    with open(inputFile,'r') as input:
        gene_infoDict = {}
        for line in input:
            fileds = line.split()
            gene_name = fileds[0].split('|')[0]
            pos = int(fileds[1])
            start = int(fileds[0].split('|')[2])
            end = int(fileds[0].split('|')[3])
            cdsLength = end - start + 1
            # geneLength = int(fileds[0].split('|')[4])
            density = float(fileds[2])
            id = ':'.join([gene_name,str(cdsLength)])
            if id in filtedGeneDict:
                reldist = pos - start
                if -50 <= reldist <= 1500:
                    # divide reads at one position by average number of reads per base for this gene
                    reads = density / filtedGeneDict[id][1]
                    rangeDict[reldist] += reads
                    # how often was this position counted in the calculation
                    countDict[reldist] += 1
                else:
                    pass
            else:
                pass
    
    #########################################################
    # output data
    # sort dict
    tupledlist1 = list(rangeDict.items())
    tupledlist1.sort()
    tupledlist2 = list(countDict.items())
    tupledlist2.sort()

    fullDict = {}
    zippedlist = zip(tupledlist1, tupledlist2)
    for elem in zippedlist:
        col0 = elem[0][0]       # list0 col0 = position (K)
        col1 = elem[0][1]       # list0 col1 = norm read number 
        col2 = elem[1][1]       # list1 col1 = how often was position counted

        #normalization2 by frequnecy
        if col2 == 0:
            fullDict[col0] = 0
        else:
            fullDict[col0] = col1 / col2        
            
    # Finish output
    tupledlist = list(fullDict.items())
    tupledlist.sort()

    # output
    outFileP = open(outputFile, 'w')
        
    for elem in tupledlist:
        outFileP.write('\t'.join([str(elem[0]),str(elem[1])]) + '\n')
    outFileP.close()

In [ ]:
import os

# make folder
os.mkdir('./4.metagene-data')

In [2]:
# output name
outputName = ['FAS1-trans-rep1','FAS1-inter-rep1','FAS1-trans-rep2','FAS1-inter-rep2',
                'FAS2-trans-rep1','FAS2-inter-rep1','FAS2-trans-rep2','FAS2-inter-rep2',
                'FAS1-MPTdel-trans-rep1','FAS1-MPTdel-inter-rep1','FAS1-MPTdel-trans-rep2','FAS1-MPTdel-trans-rep3','FAS1-MPTdel-inter-rep2','FAS1-MPTdel-inter-rep3',
                'FAS2-MPTdel-trans-rep1','FAS2-MPTdel-inter-rep1','FAS2-MPTdel-trans-rep2','FAS2-MPTdel-inter-rep2',
                'GUS1-trans-rep1','GUS1-inter-rep1','GUS1-trans-rep2','GUS1-inter-rep2',
                'MES1-trans-rep1','MES1-inter-rep1','MES1-trans-rep2','MES1-inter-rep2',
                'ARC1-trans-rep1','ARC1-inter-rep1','ARC1-trans-rep2','ARC1-inter-rep2']

# run
for i in range(0,30):
    metagenePfofileToST(''.join(['3.ribo-density-data/',outputName[i],'.density.txt']),
                        ''.join(['4.metagene-data/',outputName[i],'.metegene2StartCodon.txt']),
                        cdslength=900,expression=100)